In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
import pandas as pd
import torchvision.transforms as T
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import json
import math
import pickle
import matplotlib.pyplot as plt
from IPython.display import clear_output
from scipy.ndimage import gaussian_filter
from matplotlib.gridspec import GridSpec

from matplotlib.patches import Rectangle

from bioplnn.models import SpatiallyEmbeddedClassifier, SpatiallyEmbeddedAreaConfig, SpatiallyEmbeddedRNN
from bioplnn.datasets import Mazes
from bioplnn.utils import (
    initialize_dataloader,
)

maze_data_path = "./data/mazes"
checkpoint_path = "./train/checkpoints/"

# Torch setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

batch_size = 20

In [ ]:
def prepare_rnn_weights(state_dict):
    """Remove 'rnn.' prefix from all keys in the state dict.
        Also remove any key starting with readout"""
    new_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('rnn.'):
            new_key = key[4:]  # Remove 'rnn.' prefix
            new_state_dict[new_key] = value
        elif not key.startswith('readout'):
            new_state_dict[key] = value
    return new_state_dict
    

In [ ]:
def combine_activations(activations, abs=True):
    if abs:
        return torch.mean(torch.abs(activations), dim=1)
    else:
        return torch.mean(activations, dim=1)

def visualize_activations(excitatory_activations, inhibitory_activations, maze):
    excitatory_activations = combine_activations(excitatory_activations)
    inhibitory_activations = combine_activations(inhibitory_activations)
    
    combined_activations = torch.sum(excitatory_activations - inhibitory_activations, dim = 0)
    return combined_activations

In [ ]:
train_loader, test_loader = initialize_dataloader(
    seed=42, root="./data/mazes/", batch_size=batch_size, dataset="mazes"
)
inputs, labels = next(iter(test_loader))
mazes = Mazes(maze_data_path)

# Start Here

In [ ]:
def load_model_and_config(wandb_name, checkpoint_path):
    """Load model configuration and instantiate models."""
    try:
        full_cfg = pickle.load(open(checkpoint_path + f"{wandb_name}.pkl", "rb"))
        model_cfg, num_steps = full_cfg["model_config"], full_cfg["num_steps"]
    except:
        model_cfg = pickle.load(open(checkpoint_path + f"{wandb_name}.pkl", "rb"))
        num_steps = 20

    model = SpatiallyEmbeddedRNN(**model_cfg["rnn_kwargs"])
    classifier = SpatiallyEmbeddedClassifier(**model_cfg)
    mazes = Mazes(maze_data_path)
    
    return model, classifier, mazes, num_steps


def extract_weights(state_dict, excitatory, output):
    """Extract weight matrices and tau constants from state dict."""
    if excitatory:
        conv0 = state_dict["rnn.areas.0.convs.0->0.0.weight"].detach().cpu().numpy()
        e2e = state_dict["rnn.areas.0.convs.1->0.0.weight"].detach().cpu().numpy()
        taus = state_dict["rnn.areas.0.tau.0"].detach().cpu().numpy()[0, :, 0, 0]
    else:
        if output:
            e2e = state_dict["rnn.areas.0.convs.2->0.0.weight"].detach().cpu().numpy()
        else:
            e2e = state_dict["rnn.areas.0.convs.1->1.0.weight"].detach().cpu().numpy()
        conv0 = state_dict["rnn.areas.0.convs.0->1.0.weight"].detach().cpu().numpy()
        taus = state_dict["rnn.areas.0.tau.1"].detach().cpu().numpy()[0, :, 0, 0]
    
    return conv0, e2e, taus

def extract_output_weights(state_dict):
    """Extract weight matrices and tau constants from state dict."""
    e2e = state_dict["rnn.areas.0.out_convs.1->out.0.weight"].detach().cpu().numpy()
    return e2e


def apply_gaussian_filter(data, sigma):
    """Apply Gaussian filter to data and return filtered data with min/max values."""
    filtered_data = gaussian_filter(data, sigma=sigma)
    return filtered_data, filtered_data.min(), filtered_data.max()


def create_activity_subplot(fig, subplot_spec, data, step, title, vmin=None, vmax=None, cmap='viridis'):
    """Create a standardized activity subplot."""
    ax = fig.add_subplot(subplot_spec)
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.axis('off')
    ax.set_title(title, fontsize=8)
    return ax, im


def highlight_self_kernel(ax, is_self=False):
    """Highlight the 'self' kernel with red border or hide borders for others."""
    if is_self:
        for side in ax.spines:
            spine = ax.spines[side]
            spine.set_visible(True)
            spine.set_edgecolor('red')
            spine.set_linewidth(1.5)
    else:
        for spine in ax.spines.values():
            spine.set_visible(False)


def plot_maze_sample(fig, outer_grid, row_idx, maze_data, sample_idx, is_correct):
    """Plot maze sample with correctness indicator."""
    ax = fig.add_subplot(outer_grid[row_idx, 0])
    ax.imshow(maze_data)
    ax.set_title(f"Sample {sample_idx} {' ✓' if is_correct else ' ✗'}", fontsize=10)
    ax.axis("off")


def plot_mean_activity_grid(fig, outer_grid, row_idx, col_idx, mean_activity, step_indices, n, m):
    """Plot mean activity across all channels in a grid layout."""
    mean_sub = outer_grid[row_idx, col_idx].subgridspec(
        n, m, height_ratios=[1]*n, hspace=0, wspace=0.02
    )
    
    mean_vmin, mean_vmax = mean_activity.min(), mean_activity.max()
    mean_ims, mean_axs = [], []
    
    for k, step in enumerate(step_indices):
        r, c = divmod(k, m)
        ax, im = create_activity_subplot(
            fig, mean_sub[r, c], mean_activity[step], 
            step, f"S{step}", mean_vmin, mean_vmax
        )
        mean_ims.append(im)
        mean_axs.append(ax)
    
    # Add column title
    ax_title = fig.add_subplot(outer_grid[0, col_idx])
    ax_title.axis('off')
    ax_title.set_title("Mean Activity", fontsize=12, pad=20)
    
    return mean_ims, mean_axs


def plot_channel_activity(fig, outer_grid, row_idx, col_idx, activity, channel_idx, 
                         step_indices, n, m, input_filter, tau, kernels, sigma_filter):
    """Plot activity for a single channel including activity panels, input filter, and kernels."""
    # Setup subgrid for this channel
    K = kernels.shape[0]
    kern_cols = m
    kern_rows = math.ceil(K / kern_cols)
    total_rows = n + 1 + kern_rows
    
    sub = outer_grid[row_idx, col_idx].subgridspec(
        total_rows, m,
        height_ratios=[1]*n + [0.5] + [0.5]*kern_rows,
        hspace=0.2, wspace=0.02
    )

    activity = torch.clamp(activity, max=3)
    
    # Activity panels
    vmin, vmax = activity[:, channel_idx].flatten().min(), activity[:, channel_idx].flatten().max()
    ims, axs = [], []
    
    for k, step in enumerate(step_indices):
        r, c = divmod(k, m)
        ax, im = create_activity_subplot(
            fig, sub[r, c], activity[step, channel_idx], 
            step, f"S{step}", vmin, vmax
        )
        ims.append(im)
        axs.append(ax)
    
    # Add colorbar for activity panels
    if axs:
        plt.colorbar(ims[-1], ax=axs[-1], fraction=0.046, pad=0.04)
    
    # Column title
    ax_title = fig.add_subplot(outer_grid[0, col_idx])
    ax_title.axis('off')
    ax_title.set_title(f"Channel {channel_idx}", fontsize=12, pad=20)
    
    # Input filter
    filtered_filter, wf_vmin, wf_vmax = apply_gaussian_filter(input_filter, sigma_filter)
    ax_filter = fig.add_subplot(sub[n, :])
    im_filter = ax_filter.imshow(filtered_filter, cmap='viridis', vmin=wf_vmin, vmax=wf_vmax)
    ax_filter.set_title(f"τ={tau:.3f}", fontsize=10)
    ax_filter.axis('off')
    
    # e→e kernels
    filtered_kernels = [gaussian_filter(kernel, sigma=sigma_filter) for kernel in kernels]
    wvmin, wvmax = min(k.min() for k in filtered_kernels), max(k.max() for k in filtered_kernels)
    
    kernel_ims, kernel_axs = [], []
    for k, kernel in enumerate(filtered_kernels):
        rr = n + 1 + (k // kern_cols)
        cc = k % kern_cols
        ax_kernel = fig.add_subplot(sub[rr, cc])
        im_kernel = ax_kernel.imshow(kernel, cmap='viridis', vmin=wvmin, vmax=wvmax)
        ax_kernel.set_xticks([])
        ax_kernel.set_yticks([])
        
        # Highlight self kernel
        if do_highlight_self_kernel:
            highlight_self_kernel(ax_kernel, k == channel_idx)
        
        kernel_ims.append(im_kernel)
        kernel_axs.append(ax_kernel)
    
    return ims, axs, kernel_ims, kernel_axs

# Evaluate

In [ ]:
# ——— User‐set paths & names ———
# Identity
# wandb_name    = "twilight-spaceship-341"
# checkpoint    = 390    

# wandb_name = "happy-microwave-331"
# checkpoint = 200

# ReLU
wandb_name = "earnest-planet-436"
checkpoint = 800

# Identity, Center-Excitation
# wandb_name = "rose-elevator-396"
# checkpoint = 360

# ——— Load model config & instantiate ———
model, classifier, mazes, num_steps = load_model_and_config(wandb_name, checkpoint_path)

# ——— Load checkpoint & weights ———
try:
    state_dict = torch.load(checkpoint_path + f"{wandb_name}_{checkpoint}.pth", map_location=torch.device('cpu'))
except:
    state_dict = torch.load(checkpoint_path + f"{wandb_name}/{checkpoint}.pth", map_location=torch.device('cpu'))  

classifier.load_state_dict(state_dict)

In [ ]:
classifier.to(device)
classifier.eval()
total_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for x, labels in tqdm(test_loader):
        x = x.to(device)
        labels = labels.to(device)
        
        logits = classifier(x, num_steps=20)
        
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(accuracy)

# Plot Everything

In [ ]:
# ——— User‐set paths & names ———
# Identity
# wandb_name    = "twilight-spaceship-341"
# checkpoint    = 390    

# wandb_name = "happy-microwave-331"
# checkpoint = 200

# ReLU
wandb_name = "earnest-planet-436"
checkpoint = 800

# Identity, Center-Excitation
# wandb_name = "rose-elevator-396"
# checkpoint = 360

# ——— Load model config & instantiate ———
model, classifier, mazes, num_steps = load_model_and_config(wandb_name, checkpoint_path)

# ——— Load checkpoint & weights ———
try:
    state_dict = torch.load(checkpoint_path + f"{wandb_name}_{checkpoint}.pth", map_location=torch.device('cpu'))
except:
    state_dict = torch.load(checkpoint_path + f"{wandb_name}/{checkpoint}.pth", map_location=torch.device('cpu'))  

model.load_state_dict(prepare_rnn_weights(state_dict))
classifier.load_state_dict(state_dict)

# ——— Plot settings ———
excitatory = True
input_channel = -1 # 0 is walls, -1 is starting points
plot_maze = True
n, m = 4, 4  # activity grid dims
sigma_filter = 1.0  # optional blur for weights
output = False  # if True: kernels = e2e[:, j], else: kernels = e2e[j, :]
channels_to_plot = None #[1]  # None plots all channels, or specify list like [0, 2, 5] for specific channels
samples_to_plot = [5, 6, 7, 8, 9, 10, 11, 12, 13] #None #[1, 2, 3, 4] #[1]  # None plots all samples, or specify list like [0, 2, 5] for specific samples
do_highlight_self_kernel = True

do_output_analysis = False

# ——— Extract weight‐matrices & taus ———
conv0, e2e, taus = extract_weights(state_dict, excitatory, output)
input_filters = conv0[:, input_channel]  # ⇒ (C, H, W)

# -- Cheating a bit to do output analysis --- super lazy
if do_output_analysis:
    e2e = extract_output_weights(state_dict)

# Precompute channel count and filter channels if specified
C = input_filters.shape[0]
if channels_to_plot is not None:
    # Validate channel indices
    if not all(0 <= ch < C for ch in channels_to_plot):
        raise ValueError(f"Channel indices must be between 0 and {C-1}")
    channel_indices = channels_to_plot
    C_plot = len(channel_indices)
else:
    channel_indices = list(range(C))
    C_plot = C

# Number of columns for activity: one per channel (+1 for maze)
num_cols = C_plot + 1 + (1 if plot_maze else 0)

# Filter samples if specified
if samples_to_plot is not None:
    # Validate sample indices
    if not all(0 <= s < len(inputs) for s in samples_to_plot):
        raise ValueError(f"Sample indices must be between 0 and {len(inputs)-1}")
    sample_indices = samples_to_plot
    num_inputs = len(sample_indices)
else:
    sample_indices = list(range(len(inputs)))
    num_inputs = len(inputs)

# Setup main figure
fig = plt.figure(figsize=(3 * num_cols * m, 15 * num_inputs))
# Calculate width ratios dynamically based on what we're plotting
if plot_maze:
    width_ratios = [1] + [1.5] + [2] * C_plot
else:
    width_ratios = [1.5] + [2] * C_plot

outer = fig.add_gridspec(
    num_inputs, num_cols,
    width_ratios=width_ratios,
    hspace=0.2, wspace=0.1
)

# Pre-filter kernels for consistent color scaling
full_wvmin, full_wvmax = np.inf, -np.inf
for j in channel_indices:
    kernels = e2e[:, j] if output else e2e[j, :]
    for k in range(len(kernels)):
        kernels[k] = gaussian_filter(kernels[k], sigma=sigma_filter)
    wvmin, wvmax = kernels.min(), kernels.max()
    full_wvmin = min(full_wvmin, wvmin)
    full_wvmax = max(full_wvmax, wvmax)

# Main plotting loop
for plot_idx, sample_idx in enumerate(sample_indices):
    inp, label = inputs[sample_idx], labels[sample_idx]
    # Compute activity for sample i
    out = classifier(inp.unsqueeze(0), num_steps=num_steps)
    correct = (out.argmax(dim=1).item() == label)

    output_states, neuron_states, _ = model(inp.unsqueeze(0), num_steps=num_steps)

    act = model.query_neuron_states(neuron_states, 0, 0 if excitatory else 1) \
               .detach().cpu()[0]  # shape (T, C)
    
    if do_output_analysis:
        act = output_states[0][0].detach().cpu()

    T = act.shape[0]
    step_idxs = np.linspace(0, T - 1, n * m, dtype=int)

    # Plot maze if desired
    col_offset = 0
    if plot_maze:
        plot_maze_sample(fig, outer, plot_idx, mazes.tensor_to_image(inp), sample_idx, correct)
        col_offset = 1
    
    # Compute and plot mean activity
    mean_act = torch.mean(act, dim=1)
    plot_mean_activity_grid(fig, outer, plot_idx, col_offset, mean_act, step_idxs, n, m)
    col_offset += 1

    # Plot each channel
    for channel_plot_idx, j in enumerate(channel_indices):
        col = channel_plot_idx + col_offset
        kernels = e2e[:, j] if output else e2e[j, :]
        
        plot_channel_activity(
            fig, outer, plot_idx, col, act, j, step_idxs, n, m,
            input_filters[j], taus[j], kernels, sigma_filter
        )

plt.tight_layout()
plt.show()

# Just Kernels

In [ ]:
# ——— User‐set paths & names ———
# Identity
# wandb_name    = "twilight-spaceship-341"
# checkpoint    = 390    

# wandb_name = "happy-microwave-331"
# checkpoint = 200

# ReLU
# wandb_name = "celestial-frost-412"
# checkpoint = 400

# Identity, Center-Excitation
wandb_name = "rose-elevator-396"
checkpoint = 360

# ——— Load model config & instantiate ———
model, classifier, mazes, num_steps = load_model_and_config(wandb_name, checkpoint_path)

# ——— Load checkpoint & weights ———
try:
    state_dict = torch.load(checkpoint_path + f"{wandb_name}_{checkpoint}.pth", map_location=torch.device('cpu'))
except:
    state_dict = torch.load(checkpoint_path + f"{wandb_name}/{checkpoint}.pth", map_location=torch.device('cpu'))  

model.load_state_dict(prepare_rnn_weights(state_dict))
classifier.load_state_dict(state_dict)

# ——— Plot settings ———
excitatory = False
input_channel = -1 # 0 is walls, -1 is starting points
plot_maze = True
n, m = 2, 2  # activity grid dims
sigma_filter = 1.0  # optional blur for weights
output = True  # if True: kernels = e2e[:, j], else: kernels = e2e[j, :]
channels_to_plot = None #[1]  # None plots all channels, or specify list like [0, 2, 5] for specific channels
samples_to_plot = [1]  # None plots all samples, or specify list like [0, 2, 5] for specific samples
do_highlight_self_kernel = True

# ——— Extract weight‐matrices & taus ———
conv0, e2e, taus = extract_weights(state_dict, excitatory, output)

# e2e = extract_output_weights(state_dict)

# ——— Precompute filtered kernels & global vmin/vmax ———
filtered_kernels = {}
if output:
    channel_indices = np.arange(e2e.shape[1])
else:
    channel_indices = np.arange(e2e.shape[0])

for j in channel_indices:
    raw_kernels = e2e[:, j] if output else e2e[j, :]
    filtered_kernels[j] = [apply_gaussian_filter(k, sigma_filter)[0] for k in raw_kernels]

# all_vals = np.concatenate([k.flatten() for ks in filtered_kernels.values() for k in ks])
# vmin, vmax = all_vals.min(), all_vals.max()

n_channels = len(channel_indices)
# 1) pick a “square” grid for channels
grid_cols = int(math.ceil(math.sqrt(n_channels)))
grid_rows = int(math.ceil(n_channels / grid_cols))

# 2) make a figure sized for that grid
fig = plt.figure(figsize=(3 * grid_cols, 5 * grid_rows))
outer = fig.add_gridspec(grid_rows, grid_cols, wspace=0.4, hspace=0.6)

for idx, ch in enumerate(channel_indices):
    row = idx // grid_cols
    col = idx % grid_cols

    # 3) inside each grid‐cell: 2 rows (title + kernels)
    inner = outer[row, col].subgridspec(2, 1, height_ratios=[0.3, 6], hspace=0.1)

    # --- title ---
    ax_title = fig.add_subplot(inner[0])
    ax_title.text(
        0.5, 0.5,
        f"Channel {ch}",
        ha='center', va='center',
        fontsize=12
    )
    ax_title.axis('off')

    # --- kernels grid ---
    kernels = filtered_kernels[ch]
    K = len(kernels)
    k_cols = int(math.ceil(math.sqrt(K)))
    k_rows = int(math.ceil(K / k_cols))

    all_vals = np.concatenate([k.flatten() for k in kernels])
    vmin, vmax = all_vals.min(), all_vals.max()

    kernel_spec = inner[1].subgridspec(k_rows, k_cols, wspace=0.1, hspace=0.1)
    for k_idx, kernel in enumerate(kernels):
        r, c = divmod(k_idx, k_cols)
        ax = fig.add_subplot(kernel_spec[r, c])
        ax.imshow(kernel, cmap='viridis', vmin=vmin, vmax=vmax)
        highlight_self_kernel(ax, is_self=(k_idx == ch))
        ax.axis('off')

fig.suptitle("Kernels for Each Channel", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
state_dict.keys()

# Just activity

In [ ]:
def plot_channel_activity_solo(
    activity,       # array of shape [T, C, H, W]
    channel_idx,    # which channel to plot
    n, m,           # grid layout: n rows, m cols
    cmap='viridis'  # optional colormap
):
    """
    Standalone plot of just the activity panels for one channel.
    """
    T = activity.shape[0]
    step_indices = np.linspace(0, T - 1, n * m, dtype=int)

    # compute shared color‐scale over all selected frames
    vmin = torch.min(activity[:, channel_idx])
    vmax = torch.max(activity[:, channel_idx])
    # make figure + gridspec
    fig = plt.figure(figsize=(m * 2, n * 2))
    grid = fig.add_gridspec(n, m, hspace=0.2, wspace=0.02)
    fig.suptitle(f"Channel {channel_idx}", fontsize=14, y=0.95)

    for idx, step in enumerate(step_indices):
        r, c = divmod(idx, m)
        ax = fig.add_subplot(grid[r, c])
        im = ax.imshow(activity[step, channel_idx], cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f"S{step}", fontsize=10)
        ax.axis('off')

        # only add colorbar on the last panel
        if idx == len(step_indices) - 1:
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.show()

# ——— Load model config & instantiate ———
model, classifier, mazes, num_steps = load_model_and_config(wandb_name, checkpoint_path)

# ——— Load checkpoint & weights ———
state_dict = torch.load(checkpoint_path + f"{wandb_name}_{checkpoint}.pth", map_location=torch.device('cpu'))
model.load_state_dict(prepare_rnn_weights(state_dict))

# ——— Plot settings ———
excitatory = True
n, m = 4, 5  # activity grid dims
sigma_filter = 1.0  # optional blur for weights
channels_to_plot = [0, 1, 2, 3] 
sample_indices = [7]
do_highlight_self_kernel = False

# Main plotting loop
for plot_idx, sample_idx in enumerate(sample_indices):
    inp, label = inputs[sample_idx], labels[sample_idx]
    _, neuron_states, _ = model(inp.unsqueeze(0), num_steps=num_steps)
    activity = model.query_neuron_states(neuron_states, 0, 0 if excitatory else 1) \
               .detach().cpu()[0]  # shape (T, C)

    for channel_idx in channels_to_plot:
        plot_channel_activity_solo(
                    activity, channel_idx, n, m)

# E and I

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_activity_overlay_panels(
    ex_activity,     # ndarray[T, H, W]: excitatory frames
    inh_activity,    # ndarray[T, H, W]: inhibitory frames
    e_channel,
    inh_channel,
    n, m,             # ints: grid layout (n rows × m cols)
    alpha=0.5
):
    """
    Plot excitatory (green) and inhibitory (red) activity overlaid
    in a n×m grid. No return; just shows the figure.
    """
    ex_activity = ex_activity[:, e_channel]   # excitatory
    inh_activity = inh_activity[:, inh_channel]   # inhibitory, for example

    T = ex_activity.shape[0]
    step_indices = np.linspace(0, T - 1, n * m, dtype=int)

    # sanity checks
    if ex_activity.shape != inh_activity.shape:
        raise ValueError("ex_activity and inh_activity must have the same shape")
    total = len(step_indices)
    if total > n * m:
        raise ValueError(f"{total} panels won't fit in a {n}×{m} grid")
    
    # compute shared color‐scale over all selected frames
    vmin = torch.min(torch.min(ex_activity), torch.min(inh_activity))
    vmax = torch.max(torch.max(ex_activity), torch.max(inh_activity))

    # create figure + gridspec
    fig = plt.figure(figsize=(m*3, n*3))
    grid = fig.add_gridspec(n, m, hspace=0.2, wspace=0.1)
    fig.suptitle(f"e_channel: {e_channel}, inh_channel: {inh_channel}", fontsize=14, y=0.95)

    for idx, step in enumerate(step_indices):
        r, c = divmod(idx, m)
        ax = fig.add_subplot(grid[r, c])

        exc = ex_activity[step]
        inh = inh_activity[step]

        # show excitation
        im0 = ax.imshow(exc, cmap="Greens", vmin=vmin, vmax=vmax)
        # overlay inhibition
        im1 = ax.imshow(inh, cmap="Reds", vmin=vmin, vmax=vmax, alpha=alpha)

        ax.set_title(f"S{step}", fontsize=10)
        ax.axis("off")

        # only add one colorbar (here for excitation)
        if idx == total - 1:
            cbar = fig.colorbar(im0, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label("activity")

    plt.show()


# ——— Load model config & instantiate ———
model, classifier, mazes, num_steps = load_model_and_config(wandb_name, checkpoint_path)

# ——— Load checkpoint & weights ———
try:
    state_dict = torch.load(checkpoint_path + f"{wandb_name}_{checkpoint}.pth", map_location=torch.device('cpu'))
except:
    state_dict = torch.load(checkpoint_path + f"{wandb_name}/{checkpoint}.pth", map_location=torch.device('cpu'))
model.load_state_dict(prepare_rnn_weights(state_dict))

# ——— Plot settings ———
n, m = 4, 5  # activity grid dims
e_channels = np.arange(0, 8)
inh_channels = np.arange(0, 4)
sample_indices = [7]
do_highlight_self_kernel = False

# Main plotting loop
for plot_idx, sample_idx in enumerate(sample_indices):
    inp, label = inputs[sample_idx], labels[sample_idx]
    _, neuron_states, _ = model(inp.unsqueeze(0), num_steps=num_steps)
    exc = model.query_neuron_states(neuron_states, 0, 0) \
                .detach().cpu()[0]  # shape (T, C)
    inh = model.query_neuron_states(neuron_states, 0, 1) \
                .detach().cpu()[0]  # shape (T, C)

    for e_channel in e_channels:
        for inh_channel in inh_channels:
            plot_activity_overlay_panels(
                ex_activity=exc,
                inh_activity=inh,
                e_channel = e_channel,
                inh_channel = inh_channel,
                n=n, m=m
            )